# RQ2 RP-Geo — Stage-C finalize recovery (T4 x2)

Recovery-only notebook for a Save & Run whose RP-Geo training completed but finalization failed because one or more read-only variance diagnostics were absent. This notebook never calls a training function.

## Required inputs

1. The output of the failed Stage-C Save & Run, containing `e2e_pairwise_pilot_v2/resource_geo/checkpoints/epoch_100.pt`.
2. CIFAR-100 containing `cifar-100-python/{train,test,meta}`.
3. Gate-A output containing `gate_a_summary.json`.
4. Kaggle secret `github_token`. Enable T4 x2.

Do not attach an older Stage-C output at the same time.

In [ ]:
import os, subprocess, sys, json, time, zipfile, importlib, hashlib, shutil
from pathlib import Path
from IPython.display import display
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret('github_token')
assert github_token, 'Missing Kaggle secret github_token'
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy(); env.update({'GIT_ASKPASS':str(askpass),'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN_RUNTIME':github_token})
try:
    command = ['git','-C',str(PROJECT_ROOT),'pull','--ff-only'] if (PROJECT_ROOT/'.git').is_dir() else ['git','clone','https://github.com/duyh80456-code/new-pruning.git',str(PROJECT_ROOT)]
    subprocess.run(command, env=env, check=True)
finally:
    askpass.unlink(missing_ok=True); github_token = None
os.chdir(PROJECT_ROOT); sys.path.insert(0, str(PROJECT_ROOT))
import torch
assert torch.cuda.device_count() == 2, f'Select T4 x2; detected {torch.cuda.device_count()}'
GPU_IDS = (0,1)
GIT_COMMIT = subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True,check=True).stdout.strip()
print('Commit:', GIT_COMMIT)

## Restore exactly the completed Stage-C tree

Selection is based on the frozen RP-Geo epoch-100 checkpoint, not merely on a similarly named folder.

In [ ]:
source_candidates = [Path('/kaggle/working/new-pruning')] + [path.parent for path in Path('/kaggle/input').rglob('rq2_e2e_pairwise_pilot.py')]
source_candidates = list(dict.fromkeys(path.resolve() for path in source_candidates if (path/'rq2_e2e_pairwise_pilot.py').is_file() and (path/'scripts/run_e2e_pairwise_pilot.py').is_file()))
assert source_candidates, 'Source code not found. Run the clone/setup cell first or attach a prior notebook output containing new-pruning.'
SOURCE_CODE_ROOT = Path('/kaggle/working/new-pruning').resolve() if Path('/kaggle/working/new-pruning/rq2_e2e_pairwise_pilot.py').is_file() else source_candidates[0]
os.chdir(SOURCE_CODE_ROOT)
if str(SOURCE_CODE_ROOT) not in sys.path: sys.path.insert(0, str(SOURCE_CODE_ROOT))
import rq2_e2e_pairwise_pilot as pilot
import scripts.run_e2e_pairwise_pilot as runner
pilot = importlib.reload(pilot); runner = importlib.reload(runner)
INPUT_ROOT = Path('/kaggle/input')
DATASET_ROOT = pilot.find_cifar100_root(INPUT_ROOT)
GATE_A_SUMMARY = pilot.find_gate_a_summary(INPUT_ROOT)
search_roots = [INPUT_ROOT]
matching_archives = []
for archive in INPUT_ROOT.rglob('*.zip'):
    try:
        with zipfile.ZipFile(archive) as bundle:
            names = bundle.namelist()
            if any(name.endswith('e2e_pairwise_pilot_v2/resource_geo/checkpoints/epoch_100.pt') or name.endswith('e2e_pairwise_pilot_v2/resource_geo/latest.pt') for name in names): matching_archives.append(archive)
    except (OSError, zipfile.BadZipFile):
        pass
for index, archive in enumerate(matching_archives):
    destination = Path(f'/kaggle/working/materialized-stage-c-recovery-{index}')
    search_roots.append(pilot._safe_extract(archive, destination))
raw_roots = set()
for search_root in search_roots:
    for checkpoint in search_root.rglob('epoch_100.pt'):
        if checkpoint.parent.name == 'checkpoints' and checkpoint.parent.parent.name == 'resource_geo': raw_roots.add(checkpoint.parents[2])
    for latest in search_root.rglob('latest.pt'):
        if latest.parent.name == 'resource_geo': raw_roots.add(latest.parents[1])
audit = []
candidates = []
for root in sorted(raw_roots):
    rpgeo_checkpoint = root/'resource_geo/checkpoints/epoch_100.pt'
    rpgeo_latest = root/'resource_geo/latest.pt'
    required = [root/'frozen_protocol.json', root/'resolved_config.yaml', root/'rpgeo_frozen_retention.json', root/'rpgeo_stage_c_protocol.json', root/'common_warmup/epoch_010.pt', root/'uniform/checkpoints/epoch_100.pt', root/'resource/checkpoints/epoch_100.pt', root/'pure_sw/checkpoints/epoch_100.pt', root/'resource_geo/training_provenance.json']
    missing = [str(path.relative_to(root)) for path in required if not path.is_file()]
    if not rpgeo_checkpoint.is_file() and not rpgeo_latest.is_file(): missing.append('resource_geo/{checkpoints/epoch_100.pt or latest.pt}')
    audit.append({'root':str(root),'missing':missing,'has_epoch100':rpgeo_checkpoint.is_file(),'has_latest':rpgeo_latest.is_file()})
    if not missing: candidates.append(root)
print('Stage-C discovery audit:', json.dumps(audit,indent=2))
by_fingerprint = {}
for root in candidates:
    checkpoint = root/'resource_geo/checkpoints/epoch_100.pt' if (root/'resource_geo/checkpoints/epoch_100.pt').is_file() else root/'resource_geo/latest.pt'
    fingerprint = (pilot._sha256(checkpoint), pilot._sha256(root/'rpgeo_frozen_retention.json'))
    by_fingerprint.setdefault(fingerprint, []).append(root)
assert len(by_fingerprint) == 1, f'Expected one content-unique completed Stage-C tree. Direct/ZIP roots audited above; valid={candidates}; matching_archives={matching_archives}'
SOURCE_ROOT = sorted(next(iter(by_fingerprint.values())), key=lambda p:(len(str(p)),str(p)))[0]
ROOT = Path('/kaggle/working/e2e_pairwise_pilot_v2')
shutil.copytree(SOURCE_ROOT, ROOT, dirs_exist_ok=True)
target_checkpoint = ROOT/'resource_geo/checkpoints/epoch_100.pt'
if not target_checkpoint.is_file():
    latest = ROOT/'resource_geo/latest.pt'
    payload = torch.load(latest,map_location='cpu',weights_only=False)
    assert int(payload.get('epoch',-1)) == 100 and payload.get('method') == 'resource_geo', 'latest.pt is not a completed RP-Geo E100 checkpoint'
    target_checkpoint.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(latest,target_checkpoint)
freeze = pilot.load_frozen_rpgeo_retention(ROOT)
assert float(freeze['resource_retention']) == 0.995
provenance = json.loads((ROOT/'resource_geo/training_provenance.json').read_text())
assert float(provenance['resource_retention']) == 0.995
assert provenance['retention_frozen_before_e2e'] is True
print('Code source:', SOURCE_CODE_ROOT)
print('Recovery source:', SOURCE_ROOT)
print('Working root:', ROOT)
print('CIFAR-100:', DATASET_ROOT)
print('Gate A:', GATE_A_SUMMARY)

## Complete only missing read-only diagnostics

No optimizer, scheduler, or model update occurs in this notebook.

In [ ]:
started = time.perf_counter()
missing_base = tuple(job for job in pilot.DIAGNOSTIC_STATES if job[0] != 'common_warmup' and not (ROOT/'diagnostics'/f'{job[0]}_E{job[1]}'/'variance.csv').is_file())
missing_frozen = tuple(job for job in (('common_warmup',10),('resource_geo',50),('resource_geo',100)) if not (ROOT/'diagnostics_rpgeo_frozen'/f'{job[0]}_E{job[1]}'/'variance.csv').is_file())
print('Missing base diagnostics:', missing_base)
print('Missing frozen diagnostics:', missing_frozen)
runtime_parts = []
if missing_base:
    runtime_parts.append(runner.run_diagnostics(ROOT, DATASET_ROOT, GATE_A_SUMMARY, gpu_ids=GPU_IDS, jobs=missing_base))
if missing_frozen:
    runtime_parts.append(runner.run_diagnostics(ROOT, DATASET_ROOT, GATE_A_SUMMARY, gpu_ids=GPU_IDS, jobs=missing_frozen, output_namespace='diagnostics_rpgeo_frozen'))
decision = pilot.finalize_rpgeo_extension(ROOT)
print(json.dumps(decision, indent=2))
for table in runtime_parts:
    if not table.empty: display(table)
display(__import__('pandas').read_csv(ROOT/'rpgeo_method_summary.csv'))
display(__import__('pandas').read_csv(ROOT/'rpgeo_frozen_policy_variance_t10_50_100.csv'))
print(f'Recovery completed in {(time.perf_counter()-started)/60:.1f} minutes')

## Export the complete resumable result

In [ ]:
required = ['rpgeo_frozen_retention.json','rpgeo_stage_c_protocol.json','resource_geo/checkpoints/epoch_100.pt','resource_geo/training_provenance.json','resource_geo/rpgeo_refresh_metrics.csv','rpgeo_summary.json','rpgeo_method_summary.csv','rpgeo_trajectory_variance_diagnostics.csv','rpgeo_frozen_policy_variance_t10_50_100.csv']
missing = [name for name in required if not (ROOT/name).is_file() or (ROOT/name).stat().st_size == 0]
assert not missing, f'Missing recovery artifacts: {missing}'
bundle = Path('/kaggle/working/rq2-rpgeo-frozen-e2e-v1-recovered.zip')
with zipfile.ZipFile(bundle,'w',compression=zipfile.ZIP_DEFLATED,allowZip64=True) as archive:
    for path in ROOT.rglob('*'):
        if path.is_file(): archive.write(path,Path('e2e_pairwise_pilot_v2')/path.relative_to(ROOT))
print('Persist:',bundle,f'{bundle.stat().st_size/2**30:.2f} GiB')
bundle